# So sánh các giải pháp hỗ trợ cảnh báo nguy cơ bỏ học

Notebook tổng hợp kết quả Machine Learning và Rule-based Scoring trên hai nguồn dữ liệu. Mọi kết luận định lượng được đưa ra riêng trong từng nguồn; không dùng kết quả của Nguồn 1 để so sánh trực tiếp với kết quả của Nguồn 2.

Thứ tự ưu tiên cho mục tiêu cảnh báo sớm là Recall, F2-score, PR-AUC, F1-score và Accuracy. Các tiêu chí định tính được trình bày riêng để tránh trộn nhận định chủ quan vào metric kiểm thử.

## Cấu hình môi trường và đường dẫn

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Cấu hình hiển thị dùng chung cho bảng và biểu đồ.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

# Hỗ trợ chạy notebook từ thư mục notebooks hoặc thư mục gốc của kho mã.
root_candidates = [Path(".."), Path(".")]
PROJECT_ROOT = next(
    (path.resolve() for path in root_candidates if (path / "outputs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục outputs của dự án.")

OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_LABELS = {
    "student_dropout_and_success": "student_dropout_and_success.csv",
    "student_dropout": "student_dropout.csv",
}
SOLUTION_LABELS = {
    "ml": "Machine Learning",
    "rule_based": "Rule-based Scoring",
}

print("Thư mục dự án:", PROJECT_ROOT)
print("Thư mục biểu đồ:", CHART_DIR)

## Nạp kết quả của bốn thí nghiệm

Notebook chỉ đọc kết quả đã được chốt. Không huấn luyện lại mô hình, không thay đổi bộ luật và không điều chỉnh ngưỡng dựa trên bảng so sánh này.

In [ ]:
ML_METRICS_PATH = OUTPUT_DIR / "ml_test_metrics.csv"
RULE_METRICS_PATH = OUTPUT_DIR / "rule_based_test_metrics.csv"
RULE_CONFIG_PATH = OUTPUT_DIR / "rules" / "rule_based_config.json"

for required_path in [ML_METRICS_PATH, RULE_METRICS_PATH, RULE_CONFIG_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Thiếu file đầu vào: {required_path}")

ml_metrics = pd.read_csv(ML_METRICS_PATH)
rule_metrics = pd.read_csv(RULE_METRICS_PATH)
with RULE_CONFIG_PATH.open(encoding="utf-8") as file:
    rule_config = json.load(file)

# Chuẩn hóa tên cột để hai giải pháp có thể ghép thành cùng một bảng.
ml_metrics = ml_metrics.copy()
ml_metrics["threshold_unit"] = "probability"
ml_metrics["threshold_raw"] = np.nan

rule_metrics = rule_metrics.copy()
rule_metrics["model"] = "rule_set"
rule_metrics["threshold"] = rule_metrics.apply(
    lambda row: row["threshold_raw"]
    / rule_config[row["data_source"]]["max_score"],
    axis=1,
)
rule_metrics["threshold_unit"] = "normalized_rule_score"

comparison_columns = [
    "data_source", "solution_type", "model", "threshold",
    "threshold_unit", "threshold_raw", "precision", "recall",
    "f1", "f2", "accuracy", "roc_auc", "pr_auc",
    "tn", "fp", "fn", "tp",
]
solution_comparison = pd.concat(
    [ml_metrics[comparison_columns], rule_metrics[comparison_columns]],
    ignore_index=True,
).sort_values(["data_source", "solution_type"], ignore_index=True)

assert len(solution_comparison) == 4
assert solution_comparison.groupby("data_source")["solution_type"].nunique().eq(2).all()

print(solution_comparison.round(4).to_string(index=False))

## Kiểm tra tính công bằng của phép so sánh

Hai giải pháp chỉ được so sánh trực tiếp khi chúng có cùng `row_id` và cùng nhãn thật `y_true` trong một nguồn. Phần kiểm tra này tạo bằng chứng độc lập trước khi tính chênh lệch metric.

In [ ]:
PREDICTIONS = {}
fairness_rows = []

for source_name in SOURCE_LABELS:
    ml_path = PREDICTION_DIR / f"{source_name}_ml.csv"
    rule_path = PREDICTION_DIR / f"{source_name}_rule_based.csv"
    if not ml_path.exists() or not rule_path.exists():
        raise FileNotFoundError(f"Thiếu file dự đoán của {source_name}.")

    ml_prediction = pd.read_csv(ml_path)
    rule_prediction = pd.read_csv(rule_path)
    assert ml_prediction["row_id"].is_unique
    assert rule_prediction["row_id"].is_unique

    # Ghép theo row_id với quan hệ một-một để phát hiện dòng thiếu hoặc trùng.
    paired = ml_prediction[["row_id", "y_true", "y_pred"]].merge(
        rule_prediction[["row_id", "y_true", "y_pred"]],
        on="row_id",
        how="outer",
        suffixes=("_ml", "_rule_based"),
        indicator=True,
        validate="one_to_one",
    )
    same_rows = bool((paired["_merge"] == "both").all())
    same_labels = bool((paired["y_true_ml"] == paired["y_true_rule_based"]).all())
    assert same_rows and same_labels, (
        f"{source_name}: hai giải pháp không dùng cùng test set."
    )

    PREDICTIONS[(source_name, "ml")] = ml_prediction
    PREDICTIONS[(source_name, "rule_based")] = rule_prediction
    fairness_rows.append(
        {
            "data_source": source_name,
            "test_rows": len(paired),
            "dropout_rows": int(paired["y_true_ml"].sum()),
            "same_row_ids": same_rows,
            "same_y_true": same_labels,
            "comparison_is_valid": same_rows and same_labels,
        }
    )

fairness_audit = pd.DataFrame(fairness_rows)
print(fairness_audit.to_string(index=False))

## So sánh chênh lệch metric trong từng nguồn

Giá trị `ml_minus_rule_based` dương nghĩa là Machine Learning cao hơn; giá trị âm nghĩa là Rule-based Scoring cao hơn. Chênh lệch chỉ được diễn giải trong cùng một nguồn.

In [ ]:
METRICS = ["precision", "recall", "f1", "f2", "accuracy", "roc_auc", "pr_auc"]

metric_long = solution_comparison.melt(
    id_vars=["data_source", "solution_type"],
    value_vars=METRICS,
    var_name="metric",
    value_name="value",
)
metric_difference = (
    metric_long.pivot(
        index=["data_source", "metric"],
        columns="solution_type",
        values="value",
    )
    .reset_index()
)
metric_difference["ml_minus_rule_based"] = (
    metric_difference["ml"] - metric_difference["rule_based"]
)
metric_difference["higher_solution"] = np.select(
    [
        metric_difference["ml_minus_rule_based"] > 0,
        metric_difference["ml_minus_rule_based"] < 0,
    ],
    ["ml", "rule_based"],
    default="equal",
)

print(metric_difference.round(4).to_string(index=False))

## Lựa chọn theo mục tiêu cảnh báo sớm

Giải pháp ưu tiên được xác định theo thứ tự Recall, F2, PR-AUC, F1 và Accuracy. Đây là lựa chọn theo mục tiêu giảm bỏ sót, không có nghĩa giải pháp còn lại không còn giá trị vận hành.

In [ ]:
# Sắp xếp riêng trong từng nguồn theo đúng thứ tự ưu tiên đã thống nhất.
priority_selection = (
    solution_comparison.sort_values(
        ["data_source", "recall", "f2", "pr_auc", "f1", "accuracy"],
        ascending=[True, False, False, False, False, False],
    )
    .groupby("data_source", as_index=False)
    .first()
)

print(
    priority_selection[["data_source", "solution_type", "recall", "f2", "pr_auc"]]
    .round(4)
    .to_string(index=False)
)

## Biểu đồ metric chính

Biểu đồ chỉ thể hiện Precision, Recall, F2 và Accuracy để tránh quá tải thông tin. ROC-AUC và PR-AUC vẫn được giữ trong bảng CSV.

In [ ]:
CORE_METRICS = ["precision", "recall", "f2", "accuracy"]
CORE_LABELS = {
    "precision": "Precision",
    "recall": "Recall",
    "f2": "F2-score",
    "accuracy": "Accuracy",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for axis, source_name in zip(axes, SOURCE_LABELS):
    plot_frame = solution_comparison.loc[
        solution_comparison["data_source"] == source_name,
        ["solution_type"] + CORE_METRICS,
    ].melt(id_vars="solution_type", var_name="metric", value_name="value")
    plot_frame["solution"] = plot_frame["solution_type"].map(SOLUTION_LABELS)
    plot_frame["metric_label"] = plot_frame["metric"].map(CORE_LABELS)

    sns.barplot(
        data=plot_frame, x="metric_label", y="value", hue="solution",
        palette=["#2563EB", "#F59E0B"], ax=axis,
    )
    axis.set_title(SOURCE_LABELS[source_name], fontsize=12, fontweight="bold")
    axis.set_xlabel("Tiêu chí")
    axis.set_ylabel("Giá trị" if axis is axes[0] else "")
    axis.set_ylim(0, 1.05)
    # Gỡ chú giải trong từng ô để không che các cột dữ liệu.
    legend_handles, legend_labels = axis.get_legend_handles_labels()
    axis.get_legend().remove()
    for container in axis.containers:
        axis.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

fig.suptitle("So sánh metric chính trên cùng tập kiểm thử", fontsize=14, fontweight="bold")
# Đặt một chú giải chung bên dưới để không che bất kỳ cột nào.
fig.legend(
    legend_handles, legend_labels, title="Giải pháp",
    loc="lower center", bbox_to_anchor=(0.5, -0.03), ncol=2,
)
fig.tight_layout(rect=[0, 0.10, 1, 1])
metric_chart_path = CHART_DIR / "solution_metrics_by_source.png"
fig.savefig(metric_chart_path, dpi=180, bbox_inches="tight")
# Chỉ gọi show khi notebook đang dùng backend tương tác; luôn đóng hình sau khi lưu.
if "agg" not in plt.get_backend().lower():
    plt.show()
plt.close(fig)
print("Đã lưu:", metric_chart_path)

## Ma trận nhầm lẫn

Ma trận nhầm lẫn cho thấy trực tiếp số sinh viên bỏ sót (`FN`) và số cảnh báo nhầm (`FP`). Với hệ thống cảnh báo sớm, FN đặc biệt quan trọng vì đây là sinh viên bỏ học nhưng không được cảnh báo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row_index, source_name in enumerate(SOURCE_LABELS):
    for column_index, solution_type in enumerate(["ml", "rule_based"]):
        prediction_frame = PREDICTIONS[(source_name, solution_type)]
        matrix = confusion_matrix(
            prediction_frame["y_true"], prediction_frame["y_pred"], labels=[0, 1]
        )
        axis = axes[row_index, column_index]
        sns.heatmap(
            matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Không bỏ học", "Bỏ học"],
            yticklabels=["Không bỏ học", "Bỏ học"], ax=axis,
        )
        axis.set_title(
            f"{SOURCE_LABELS[source_name]}\n{SOLUTION_LABELS[solution_type]}",
            fontsize=11, fontweight="bold",
        )
        axis.set_xlabel("Dự đoán")
        axis.set_ylabel("Thực tế")

fig.suptitle("Ma trận nhầm lẫn của bốn thí nghiệm", fontsize=14, fontweight="bold")
fig.tight_layout()
confusion_chart_path = CHART_DIR / "solution_confusion_matrices.png"
fig.savefig(confusion_chart_path, dpi=180, bbox_inches="tight")
# Chỉ gọi show khi notebook đang dùng backend tương tác; luôn đóng hình sau khi lưu.
if "agg" not in plt.get_backend().lower():
    plt.show()
plt.close(fig)
print("Đã lưu:", confusion_chart_path)

## So sánh định tính

Bảng định tính mô tả đặc điểm phương pháp, không phải kết quả benchmark về thời gian hay chi phí phần cứng. Các nhận định cần được cập nhật nếu kiến trúc triển khai thay đổi.

In [ ]:
qualitative_comparison = pd.DataFrame(
    [
        {"criterion": "Khả năng giải thích từng cảnh báo", "machine_learning": "Trung bình; có mức độ quan trọng đặc trưng nhưng cần diễn giải thêm", "rule_based": "Cao; điều kiện, điểm và khuyến nghị được trả về trực tiếp"},
        {"criterion": "Chi phí xây dựng ban đầu", "machine_learning": "Cần tiền xử lý, huấn luyện và lựa chọn mô hình", "rule_based": "Cần chuyên gia thiết kế và kiểm chứng luật"},
        {"criterion": "Chi phí cập nhật", "machine_learning": "Cần dữ liệu mới và huấn luyện, đánh giá lại", "rule_based": "Có thể sửa luật trực tiếp nhưng vẫn phải đánh giá lại"},
        {"criterion": "Khả năng học quan hệ từ dữ liệu", "machine_learning": "Cao hơn; trọng số được học từ train set", "rule_based": "Hạn chế trong các điều kiện và trọng số đã khai báo"},
        {"criterion": "Tính nhất quán của quyết định", "machine_learning": "Cao khi pipeline và phiên bản mô hình được cố định", "rule_based": "Cao khi cấu hình luật và ngưỡng được cố định"},
        {"criterion": "Xử lý trường hợp ngoài dự kiến", "machine_learning": "Phụ thuộc độ bao phủ của dữ liệu huấn luyện", "rule_based": "Không phát hiện được tín hiệu chưa được viết thành luật"},
        {"criterion": "Tạo nguyên nhân và khuyến nghị cá nhân", "machine_learning": "Cần thêm tầng giải thích và ánh xạ khuyến nghị", "rule_based": "Có sẵn trong từng điều kiện luật"},
        {"criterion": "Phụ thuộc chất lượng dữ liệu", "machine_learning": "Cao", "rule_based": "Vẫn cao nhưng dễ phát hiện trường thiếu bắt buộc hơn"},
        {"criterion": "Tốc độ suy luận kỳ vọng", "machine_learning": "Nhanh với các mô hình hiện tại; chưa benchmark", "rule_based": "Rất nhanh do chỉ kiểm tra điều kiện; chưa benchmark"},
    ]
)

print(qualitative_comparison.to_string(index=False))

## Kết luận riêng cho từng nguồn

Kết luận được tạo từ chênh lệch metric thực tế. Kiến trúc kết hợp được đề xuất vì hai giải pháp có ưu điểm bổ sung: Machine Learning cung cấp khả năng xếp hạng từ dữ liệu, còn Rule-based cung cấp lý do và hành động can thiệp rõ ràng.

In [ ]:
recommendation_rows = []
for source_name in SOURCE_LABELS:
    source_metrics = solution_comparison.set_index(["data_source", "solution_type"])
    ml_row = source_metrics.loc[(source_name, "ml")]
    rule_row = source_metrics.loc[(source_name, "rule_based")]
    winner = priority_selection.loc[
        priority_selection["data_source"] == source_name, "solution_type"
    ].iloc[0]

    # Diễn giải chênh lệch bằng điểm phần trăm để nhóm dễ trình bày.
    recall_difference = (ml_row["recall"] - rule_row["recall"]) * 100
    f2_difference = (ml_row["f2"] - rule_row["f2"]) * 100
    precision_difference = (ml_row["precision"] - rule_row["precision"]) * 100
    accuracy_difference = (ml_row["accuracy"] - rule_row["accuracy"]) * 100

    evidence = (
        f"ML - Rule-based: Recall {recall_difference:+.2f} điểm %, "
        f"F2 {f2_difference:+.2f} điểm %, Precision {precision_difference:+.2f} điểm %, "
        f"Accuracy {accuracy_difference:+.2f} điểm %."
    )
    if source_name == "student_dropout_and_success":
        interpretation = (
            "Machine Learning phù hợp hơn với mục tiêu hạn chế bỏ sót; "
            "Rule-based tạo ít cảnh báo nhầm hơn và giải thích trực tiếp."
        )
    else:
        interpretation = (
            "Rule-based nhỉnh hơn rất ít về Recall và F2; Machine Learning tốt hơn "
            "về Precision, Accuracy và khả năng xếp hạng. Dữ liệu mô phỏng nên chưa đủ "
            "để chọn một giải pháp duy nhất cho triển khai thực tế."
        )

    recommendation_rows.append(
        {
            "data_source": source_name,
            "priority_winner": winner,
            "quantitative_evidence": evidence,
            "interpretation": interpretation,
            "operational_recommendation": (
                "Kết hợp hai giải pháp: dùng ML để dự đoán/xếp hạng và Rule-based "
                "để giải thích, đưa khuyến nghị và làm phương án dự phòng."
            ),
        }
    )

solution_recommendations = pd.DataFrame(recommendation_rows)
print(solution_recommendations.to_string(index=False))

## Lưu báo cáo so sánh

`solution_comparison.csv` là bảng định lượng chính theo yêu cầu môn học. Các file còn lại cung cấp bằng chứng công bằng, chênh lệch metric, tiêu chí định tính và đề xuất vận hành.

In [ ]:
# Lưu bảng định lượng chính với đủ bốn thí nghiệm.
solution_comparison.to_csv(
    OUTPUT_DIR / "solution_comparison.csv", index=False, encoding="utf-8-sig"
)
metric_difference.to_csv(
    OUTPUT_DIR / "solution_metric_differences.csv",
    index=False,
    encoding="utf-8-sig",
)
qualitative_comparison.to_csv(
    OUTPUT_DIR / "solution_comparison_qualitative.csv",
    index=False,
    encoding="utf-8-sig",
)
fairness_audit.to_csv(
    OUTPUT_DIR / "comparison_fairness_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
solution_recommendations.to_csv(
    OUTPUT_DIR / "solution_recommendations.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Đã lưu bảng so sánh và đề xuất vào outputs.")

## Kiểm tra đầu ra

Phần kiểm tra cuối bảo đảm bảng chính có đủ hai nguồn, hai loại giải pháp và toàn bộ metric bắt buộc.

In [ ]:
required_comparison_columns = {
    "data_source", "solution_type", "precision", "recall",
    "f1", "f2", "accuracy", "roc_auc", "pr_auc",
}
assert required_comparison_columns.issubset(solution_comparison.columns)
assert set(solution_comparison["data_source"]) == set(SOURCE_LABELS)
assert set(solution_comparison["solution_type"]) == {"ml", "rule_based"}
assert len(solution_comparison) == 4
assert fairness_audit["comparison_is_valid"].all()
assert metric_chart_path.exists() and confusion_chart_path.exists()

print("Kiểm tra hoàn tất: bảng so sánh có đủ bốn thí nghiệm và hai biểu đồ.")

## Kết luận

Không có một giải pháp thắng tuyệt đối trên mọi tiêu chí. Với `student_dropout_and_success.csv`, Machine Learning được ưu tiên theo mục tiêu Recall trước F2; Rule-based có Precision và Accuracy cao hơn. Với `student_dropout.csv`, Rule-based nhỉnh hơn về Recall và F2, trong khi Machine Learning tốt hơn về Precision, Accuracy, ROC-AUC và PR-AUC.

Giải pháp đề xuất cho hệ thống là kiến trúc kết hợp: Machine Learning phục vụ dự đoán và xếp hạng nguy cơ; Rule-based cung cấp nguyên nhân, khuyến nghị và phương án dự phòng. Kết luận cho Nguồn 2 phải kèm giới hạn rằng dữ liệu là mô phỏng và cần được kiểm chứng trên dữ liệu thực tế.